# 배달 플랫폼 메뉴 수집·통합 품질 점검

## TL;DR

배민 기준 매칭률, 플랫폼 전용 메뉴, 가격·옵션 예외를 로컬 SQLite 원장에서 재계산합니다.

## Context & Methods

배달의민족을 기준 플랫폼으로 두고 현재 존재하는 원본 메뉴만 분석합니다. 1:1과 1:N 매칭을 분리하고 옵션 선언 연결 수와 저장 연결 수를 대조합니다.

In [1]:
import os, sqlite3, json
from pathlib import Path
DB_PATH = Path(os.environ['APPDATA']) / 'delivery-menu-sync' / 'delivery-menu-sync.db'
connection = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
connection.row_factory = sqlite3.Row
def rows(sql): return [dict(row) for row in connection.execute(sql)]
DB_PATH.name

'delivery-menu-sync.db'

## Data

In [2]:
coverage_sql = "with baemin_menus as (\n  select distinct menu_id\n  from platform_menu_mappings\n  where platform_code = 'baemin' and mapping_status = 'active'\n), target_counts as (\n  select b.menu_id, p.platform_code, count(m.platform_menu_id) as target_count\n  from baemin_menus b\n  cross join (\n    select 'coupangeats' platform_code union all\n    select 'ddangyo' union all\n    select 'deliveryspecial' union all\n    select 'yogiyo'\n  ) p\n  left join platform_menu_mappings m\n    on m.menu_id = b.menu_id\n   and m.platform_code = p.platform_code\n   and m.mapping_status = 'active'\n  group by b.menu_id, p.platform_code\n)\nselect platform_code,\n       count(*) as baemin_reference_count,\n       sum(target_count > 0) as matched_count,\n       sum(target_count = 1) as one_to_one_count,\n       sum(target_count > 1) as one_to_many_count,\n       sum(target_count = 0) as missing_count,\n       round(1.0 * sum(target_count > 0) / count(*), 4) as match_rate\nfrom target_counts\ngroup by platform_code\norder by platform_code"
coverage = rows(coverage_sql)
coverage

[{'platform_code': 'coupangeats',
  'baemin_reference_count': 46,
  'matched_count': 34,
  'one_to_one_count': 34,
  'one_to_many_count': 0,
  'missing_count': 12,
  'match_rate': 0.7391},
 {'platform_code': 'ddangyo',
  'baemin_reference_count': 46,
  'matched_count': 41,
  'one_to_one_count': 41,
  'one_to_many_count': 0,
  'missing_count': 5,
  'match_rate': 0.8913},
 {'platform_code': 'deliveryspecial',
  'baemin_reference_count': 46,
  'matched_count': 45,
  'one_to_one_count': 45,
  'one_to_many_count': 0,
  'missing_count': 1,
  'match_rate': 0.9783},
 {'platform_code': 'yogiyo',
  'baemin_reference_count': 46,
  'matched_count': 30,
  'one_to_one_count': 13,
  'one_to_many_count': 17,
  'missing_count': 16,
  'match_rate': 0.6522}]

In [3]:
catalogs = rows("select platform_code, count(*) as menu_count, max(last_seen_at) as last_seen_at\nfrom platform_menus\nwhere presence_status = 'present'\ngroup by platform_code\norder by platform_code")
options = rows("select platform_code,\n       count(*) as option_group_count,\n       sum(mapping_menus_count) as declared_binding_count,\n       sum(json_array_length(menus_json)) as stored_binding_count\nfrom platform_option_groups\nwhere presence_status = 'present'\ngroup by platform_code\norder by platform_code")
catalogs, options

([{'platform_code': 'baemin',
   'menu_count': 46,
   'last_seen_at': '2026-07-28 12:32:20'},
  {'platform_code': 'coupangeats',
   'menu_count': 38,
   'last_seen_at': '2026-07-26 03:20:34'},
  {'platform_code': 'ddangyo',
   'menu_count': 44,
   'last_seen_at': '2026-07-28 12:14:56'},
  {'platform_code': 'deliveryspecial',
   'menu_count': 47,
   'last_seen_at': '2026-07-28 08:12:06'},
  {'platform_code': 'yogiyo',
   'menu_count': 71,
   'last_seen_at': '2026-07-28 08:09:42'}],
 [{'platform_code': 'baemin',
   'option_group_count': 12,
   'declared_binding_count': 168,
   'stored_binding_count': 168},
  {'platform_code': 'coupangeats',
   'option_group_count': 11,
   'declared_binding_count': 135,
   'stored_binding_count': 135},
  {'platform_code': 'ddangyo',
   'option_group_count': 11,
   'declared_binding_count': 171,
   'stored_binding_count': 171},
  {'platform_code': 'deliveryspecial',
   'option_group_count': 24,
   'declared_binding_count': 186,
   'stored_binding_count': 1

## Results

In [4]:
platform_only = rows("select m.base_name, m.base_price,\n       group_concat(mm.platform_code || ':' || mm.platform_menu_name, ' | ') as sources,\n       count(distinct mm.platform_code) as platform_count\nfrom menus m\njoin platform_menu_mappings mm\n  on mm.menu_id = m.menu_id and mm.mapping_status = 'active'\nwhere not exists (\n  select 1 from platform_menu_mappings b\n  where b.menu_id = m.menu_id\n    and b.platform_code = 'baemin'\n    and b.mapping_status = 'active'\n)\ngroup by m.menu_id\norder by m.base_name")
len(platform_only), platform_only

(27,
 [{'base_name': '갈릭소스',
   'base_price': 500,
   'sources': 'coupangeats:갈릭소스',
   'platform_count': 1},
  {'base_name': '고구마 피자',
   'base_price': 21900,
   'sources': 'yogiyo:고구마 피자 M | yogiyo:고구마 피자 L',
   'platform_count': 1},
  {'base_name': '고르곤졸라씬피자 L',
   'base_price': 23900,
   'sources': 'yogiyo:고르곤졸라씬피자 L',
   'platform_count': 1},
  {'base_name': '로제 오븐 스파게티',
   'base_price': 10000,
   'sources': 'yogiyo:로제 오븐 스파게티',
   'platform_count': 1},
  {'base_name': '리치골드피자  L＋L',
   'base_price': 47900,
   'sources': 'yogiyo:리치골드피자  L＋L',
   'platform_count': 1},
  {'base_name': '반반피자',
   'base_price': 20900,
   'sources': 'yogiyo:반반피자 M | yogiyo:반반피자 L',
   'platform_count': 1},
  {'base_name': '베이컨 토마토 오븐스파게티',
   'base_price': 10000,
   'sources': 'yogiyo:베이컨 토마토 오븐스파게티',
   'platform_count': 1},
  {'base_name': '불고기  토마토오븐 스파게티',
   'base_price': 10000,
   'sources': 'yogiyo:불고기  토마토오븐 스파게티',
   'platform_count': 1},
  {'base_name': '불고기 아라비아따 오븐 스파게티',
   'base_price': 

In [5]:
reviews = rows("select kind, platform_code, title, recommendation, evidence_json\nfrom catalog_review_items\nwhere state = 'open'\norder by kind, platform_code, title")
from collections import Counter
Counter(row['kind'] for row in reviews)

Counter({'missing_on_platform': 34,
         'price_outlier': 7,
         'duplicate_option_group': 1})

In [6]:
assert all(row['declared_binding_count'] == row['stored_binding_count'] for row in options)
assert len(platform_only) == 27
assert sum(row['missing_count'] for row in coverage) == 34
'quality checks passed'

'quality checks passed'

## Takeaways

수집 완결성은 양호합니다. 통합 적용에서는 요기요 1:N 사이즈 구조를 보존하고, 플랫폼 전용 27개를 별칭 후보와 실제 전용 메뉴로 승인 분류해야 합니다. 가격 차이와 옵션 슬롯은 자동 병합하지 않습니다.